In [7]:
# Import necessary libraries
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import numpy as np

import swcol as sw

In [8]:
scenario_name = 'Simulation'
scenario_path = '../../../data/Colombia/scenarios/0_upme/'
model_inputs_path = scenario_path+'inputs/'
model_outputs_path = scenario_path+'outputs/'
gen_per_res_vs_marg_cost = '../../../data/XM-API/Plan/gen_per_res_vs_marg_cost/Esc0.csv'
emissions = '../../../data/XM-API/Plan/emissions/esc0.csv'
years = [2023, 2035]
cap_2023 = '../../../data/XM-API/Plan/installed_capacity/esc0/2023.csv'
cap_2037 = '../../../data/XM-API/Plan/installed_capacity/esc0/2037.csv'
dema_path = '../../../data/XM-API/variable_query/2022-12-01_2023-11-30/'

In [9]:
sw.scenarios.table(model_outputs_path, model_inputs_path)

Year,2023,2024,2025,2026,2027,2028,2029,2030,2031,2032,2033,2034,2035,2036,2037
Tech,,,,,,,,,,,,,,,
Eolica,20.00,0.0,0.0,0.0,255.0,450.0,492.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Hidro,41.89,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
RunOfRiver,3.75,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Thermal,0.00,52.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
pv_solar,466.60,1079.9,110.0,218.0,128.0,128.0,123.0,109.0,109.0,91.0,79.0,80.0,68.0,51.0,69.0
Total,532.24,1131.9,110.0,218.0,383.0,578.0,615.0,109.0,109.0,91.0,79.0,80.0,68.0,51.0,69.0


In [10]:
# Transform data
sce_sw = pd.read_csv(model_outputs_path+'/dispatch.csv')
sce_sw['year'] = sce_sw['timestamp'].str[:4]
sce_sw['Energy_TWh_typical_yr'] = sce_sw['Energy_GWh_typical_yr'] / 1000
sce_sw.rename(columns={'gen_tech': 'Tech'}, inplace=True)
sce_sw = sce_sw[sce_sw['timestamp'].str.contains('^'+str(years[0])) == False]
# Plot
sw.scenarios.dispatched_generation(sce_sw, 'Generation', 'TWh', 'year', 'Energy_TWh_typical_yr', 'Tech', scenario_name, 'Switch', 4.6)
#sce_sw = sce_sw[['generation_project','year','Energy_TWh_typical_yr','Tech']]
#sce_sw.to_csv('./9_switch.csv')

In [11]:
sce_upme = pd.read_csv(gen_per_res_vs_marg_cost)
sce_upme['Energy_TWh_typical_yr'] = sce_upme['Energy_GWh_typical_yr'] / 1000
sce_upme = sce_upme[sce_upme['year'] <= years[1]]
sw.scenarios.dispatched_generation(sce_upme, 'Generation', 'TWh', 'year', 'Energy_TWh_typical_yr', 'Tech', scenario_name, 'UPME', 4)
#sce_upme.to_csv('./9_upme.csv')

In [12]:
em_sw = pd.read_csv(model_outputs_path + '/emissions.csv')
em_sw['AnnualEmissions_MtCO2_per_yr'] = em_sw['AnnualEmissions_tCO2_per_yr'] / 1e6
em_sw = em_sw[em_sw['PERIOD'] != years[0]]
sw.scenarios.annual_emmissions(em_sw, 'PERIOD', 'AnnualEmissions_MtCO2_per_yr', scenario_name, 'Switch')

In [13]:
em_upme = pd.read_csv(emissions)
em_upme = em_upme[em_upme['PERIOD'] <= years[1]]
em_upme['AnnualEmissions_MtCO2_per_yr'] = em_upme['AnnualEmissions_tCO2_per_yr']
sw.scenarios.annual_emmissions(em_upme, 'PERIOD', 'AnnualEmissions_tCO2_per_yr', scenario_name, 'UPME')

In [14]:
sw.scenarios.annual_emissions_combined(
    dataframes=[em_sw, em_upme],
    x_axis='PERIOD',
    y_axis='AnnualEmissions_MtCO2_per_yr',
    labels=['Switch Colombia Simulation', 'Operational Simulation (UPME)'],
    colors=['#1f77b4', '#ff7f0e'],
    folder=scenario_name,
    title=scenario_name
)
#em_sw = em_sw[['PERIOD','AnnualEmissions_MtCO2_per_yr']]
#em_sw.to_csv('./10_switch.csv')
#em_upme = em_upme[['PERIOD','AnnualEmissions_MtCO2_per_yr']]
#em_upme.to_csv('./10_upme.csv')

In [15]:
import pandas as pd

cap_inst = pd.read_csv(model_outputs_path + 'BuildGen.csv')
gen_info = pd.read_csv(model_inputs_path + 'gen_info.csv')

# Unir con info de tecnologías
cap_inst = pd.merge(cap_inst, gen_info, left_on='GEN_BLD_YRS_1',
                    right_on='GENERATION_PROJECT', how='inner')
# Filtrar solo plantas activas (gen_max_age > 1)
cap_inst = cap_inst[cap_inst['gen_max_age'] > 1]
# Expandir cada planta en sus años activos
records = []
for _, row in cap_inst.iterrows():
    start_year = row['GEN_BLD_YRS_2']
    end_year = start_year + row['gen_max_age'] - 1
    for year in range(start_year, end_year + 1):
        if year <= 2035:
            records.append({
                'year': year,
                'gen_tech': row['gen_tech'],
                'BuildGen': row['BuildGen']
            })

# Crear nuevo DataFrame con capacidad activa por año
cap_inst_expanded = pd.DataFrame(records)
# Sumar capacidad instalada activa por año y tecnología
cap_inst_switch = cap_inst_expanded.groupby(['year', 'gen_tech'], as_index=False)['BuildGen'].sum()
cap_inst_switch = cap_inst_switch[cap_inst_switch['year'] >= years[0]]
cap_inst_switch = cap_inst_switch[cap_inst_switch['year'] <= years[1]]
cap_inst_switch['BuildGen'] = cap_inst_switch['BuildGen'] / 1000
sw.scenarios.dispatched_generation(cap_inst_switch, 'Installed Capacity', 'GWh', 'year', 'BuildGen', 'gen_tech', scenario_name, 'Switch', 1)

#cap_inst_switch.to_csv('./11_switch.csv')

In [16]:
cap_inst_upme = pd.read_csv('../../../data/UPME/scenarios/0_upme.csv')
all_techs  = cap_inst_upme["gen_tech"].unique()

full_index = pd.MultiIndex.from_product([range(years[0], years[1] + 1), all_techs], names=["year", "gen_tech"])
all_combinations = pd.DataFrame(index=full_index).reset_index()

cap_inst_upme = pd.merge(all_combinations, cap_inst_upme, on=["year", "gen_tech"], how="left")
cap_inst_upme["BuildGen"] = cap_inst_upme["BuildGen"].fillna(0)

cap_inst_upme["BuildGen"] = cap_inst_upme.groupby("gen_tech")["BuildGen"].cumsum()
cap_inst_upme["BuildGen"] = cap_inst_upme["BuildGen"] / 1000
sw.scenarios.dispatched_generation(cap_inst_upme, 'Installed Capacity', 'GWh', 'year', 'BuildGen', 'gen_tech', scenario_name, 'Upme', 1)
#cap_inst_upme.to_csv('./11_upme.csv')

In [17]:
cap_inst_2023 = cap_inst[cap_inst['GEN_BLD_YRS_2'] <= 2023].copy()
cap_inst_2023.rename(columns={'gen_tech': 'Tech'}, inplace=True)
cap_inst_2023 = cap_inst_2023.groupby(['Tech']).agg({
    'BuildGen': 'sum'
}).reset_index()
cap_inst_2023['BuildGen'] = cap_inst_2023['BuildGen']

cap_inst = cap_inst[cap_inst['GEN_BLD_YRS_2'] <= 2037].copy()
cap_inst.rename(columns={'gen_tech': 'Tech'}, inplace=True)

#sw.scenarios.installed_capacity(cap_inst_2023, cap_inst, '2023', '2035', scenario_name, 'Switch')

In [18]:
import pandas as pd
esc_cap_2023 = pd.read_csv(cap_2023)
esc_cap_2023.rename(columns={'gen_tech': 'Tech'}, inplace=True)
esc_cap_2037 = pd.read_csv(cap_2037)
esc_cap_2037.rename(columns={'gen_tech': 'Tech'}, inplace=True)

#sw.scenarios.installed_capacity(esc_cap_2023, esc_cap_2037, '2023', '2035', scenario_name, 'UPME')